# Inferência: InternVL3_5-2B (Configuração A, bottle)

Modelo: [OpenGVLab/InternVL3_5-2B](https://huggingface.co/OpenGVLab/InternVL3_5-2B). Ficha em `docs/models/InternVL3_5-2B/README.md`.

Protocolo igual para todos os modelos do benchmark:
- dados: `data/subset_avaliacao.json` (40 imagens do bottle, 3 perguntas = 120 respostas);
- prompt: Configuração A (imagem + `Question: ...`), sem evidência;
- geração gulosa (`do_sample=False`), até 128 tokens novos;
- métrica: acerto por gabarito (`src.multicategory.score_response`), o mesmo do exp007.

Antes de rodar, no painel lateral do Kaggle:
- **Accelerator**: GPU T4 x2 (ou P100/L4; a T4 roda em fp16).
- **Internet**: ligada (`git clone` e download dos modelos do Hugging Face).
- **Input** (opcional): dataset `ipythonx/mvtec-ad`. Sem ele, o `kagglehub` baixa (~5 GB).
- **Retomada** (opcional): anexe o Output de uma versão anterior como Input e aponte `PREVIOUS` para a pasta `bench` dela.

O código vem do GitHub: faça push de `src/benchmark.py` antes de rodar.

**Log como cópia de segurança**: cada resposta sai impressa numa linha `BENCH_ROW {json}` (e os metadados da execução em `BENCH_META`, com o hash do subconjunto). Se a sessão cair antes de salvar, baixe o log ou o notebook com saídas e recupere no repo local:

```
python -m src.benchmark recover --log <arquivo .log, .ipynb ou .txt>
```

Prefira **Save Version > Save & Run All (Commit)**: roda no servidor do Kaggle e não depende da sua conexão.

In [ ]:
import os, shutil, subprocess, sys

STORAGE = '/kaggle/working'
LEGACY_VENV = '/tmp/venv_legacy'
# Pasta bench de uma versão anterior anexada como Input, para continuar de onde parou.
PREVIOUS = None

In [ ]:
%cd /tmp
!test -d vision-explain || git clone https://github.com/Edryck/vision-explain.git
%cd /tmp/vision-explain
!git pull --ff-only
# torch e torchvision já vêm na imagem; o Qwen3.5 precisa do transformers mais novo.
!pip install -q -U transformers accelerate num2words kagglehub

## Dados

Usa o MVTec AD anexado em `/kaggle/input` se existir; senão baixa com `kagglehub`.

In [ ]:
import kagglehub

if not os.path.isdir('data/mvtec_ad/mvtec-ad/bottle'):
    path = kagglehub.dataset_download('ipythonx/mvtec-ad')
    # O dataset pode vir com pasta extra; acha onde está o bottle.
    root = os.path.dirname(next(d for d, _, _ in os.walk(path) if d.endswith('/bottle')))
    !mkdir -p data/mvtec_ad
    !rm -rf data/mvtec_ad/mvtec-ad
    !ln -s {root} data/mvtec_ad/mvtec-ad
!ls data/mvtec_ad/mvtec-ad/

Respostas e resultados ficam em `/kaggle/working/bench` via link simbólico, para sair no Output da versão.

In [ ]:
os.makedirs(f'{STORAGE}/bench', exist_ok=True)
if PREVIOUS:
    shutil.copytree(PREVIOUS, f'{STORAGE}/bench', dirs_exist_ok=True)
!rm -rf experiments/bench
!ln -s {STORAGE}/bench experiments/bench

In [ ]:
import torch
print(torch.cuda.device_count(), 'GPU(s):', torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1), 'GB')
print('bf16:', torch.cuda.is_bf16_supported())

## Execução isolada por modelo

Cada modelo roda num processo separado, o que libera a GPU entre um e outro. O InternVL usa código remoto que quebra no transformers 5.x, então roda num venv com o transformers fixado em `LEGACY_TRANSFORMERS`, herdando o torch do sistema.

In [ ]:
import json
import pandas as pd
from IPython.display import display

from src.benchmark import BASELINE, LEGACY_TRANSFORMERS, MODELS, plot_results, report, responses_path


def python_for(key):
    if MODELS[key]['env'] == 'latest':
        return sys.executable
    python = f'{LEGACY_VENV}/bin/python'
    if not os.path.exists(python):
        # Sem pip próprio: o pip do sistema instala dentro do venv mesmo assim.
        subprocess.run([sys.executable, '-m', 'venv', '--system-site-packages', '--without-pip', LEGACY_VENV], check=True)
        subprocess.run([python, '-m', 'pip', 'install', '-q', f'transformers=={LEGACY_TRANSFORMERS}', 'timm', 'einops'], check=True)
    return python


def stream(args):
    # subprocess.run escreve no terminal do kernel, não na célula; repassando linha a linha, tudo fica no log da versão.
    process = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    if process.wait():
        raise subprocess.CalledProcessError(process.returncode, args)


def run_model(key, subset, limit=None):
    args = [python_for(key), '-m', 'src.benchmark', 'run', '--model', key, '--subset', subset]
    if limit:
        args += ['--limit', str(limit)]
    stream(args)


def run_faithscore(key, subset):
    # O juiz é o SmolVLM-Instruct, que roda no transformers novo.
    stream([sys.executable, '-m', 'src.benchmark', 'faithscore', '--model', key, '--subset', subset])

## Inferência

A retomada é automática: respostas já salvas são puladas. Use `LIMIT = 2` para um teste rápido.

In [ ]:
KEY = 'InternVL3_5-2B'
LIMIT = None
# A linha de base dá a referência na tabela e no McNemar; são só 120 respostas.
RUN_BASELINE = True

In [ ]:
run_model(KEY, 'bottle', LIMIT)

In [ ]:
if RUN_BASELINE:
    run_model(BASELINE, 'bottle', LIMIT)

## Respostas

In [ ]:
record = json.load(open(responses_path('bottle', KEY)))
print(record['n_responses'], 'de', record['n_expected'], '| pico de memória:', record['peak_memory_gb'], 'GB |', record['environment'])
pd.set_option('display.max_colwidth', 200)
pd.DataFrame(record['responses'])[['image_id', 'question_id', 'answer', 'new_tokens', 'seconds']].head(30)

## Resultados

Tabela e gráficos com este modelo e a linha de base (se rodou).

In [ ]:
table, by_category = report('bottle')
shown = table[table['modelo'].isin([KEY, BASELINE])]
display(shown.style.format(precision=3))

In [ ]:
plot_results('bottle', shown, by_category[shown['modelo']])

## Exportar

Gera `/kaggle/working/bench_results.zip`. Baixe pela aba Output e extraia em `experiments/bench/` no repo local.

In [ ]:
shutil.make_archive(f'{STORAGE}/bench_results', 'zip', f'{STORAGE}/bench')
print(round(os.path.getsize(f'{STORAGE}/bench_results.zip') / 1e6, 2), 'MB')